In [15]:
import pandas as pd
import time

start = time.time()
file_path = "Crimes_2021.csv"

df = pd.read_csv(
    file_path,
    usecols=["Date", "Primary Type","Description","Location Description", "Arrest", "District", "Ward", "Community Area", "Domestic"],
    parse_dates=["Date"],
    date_format="%m/%d/%Y %I:%M:%S %p"
)


In [16]:
df.head()

,Date,Primary Type,Description,Location Description,Arrest,Domestic,District,Ward,Community Area
0,2021-07-07 10:30:00,SEX OFFENSE,SEXUAL EXPLOITATION OF A CHILD,RESIDENCE,False,False,5.0,10.0,54.0
1,2021-04-17 15:20:00,ROBBERY,VEHICULAR HIJACKING,RESIDENCE,False,False,6.0,6.0,44.0
2,2021-09-08 16:45:00,HOMICIDE,FIRST DEGREE MURDER,CAR WASH,True,False,11.0,24.0,25.0
3,2021-08-01 00:00:00,SEX OFFENSE,CRIMINAL SEXUAL ABUSE,APARTMENT,False,False,4.0,8.0,45.0
4,2021-12-10 00:00:00,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT OVER $ 300,RESIDENCE,False,False,9.0,11.0,60.0


In [17]:
df = df.dropna()

In [18]:
print(df.columns.tolist())

['Date', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'District', 'Ward', 'Community Area']


In [19]:
df["incident_hour"] = df["Date"].dt.hour
df["incident_day"] = df["Date"].dt.day_name()
df["incident_month"] = df["Date"].dt.month
df["incident_year"] = df["Date"].dt.year

In [20]:
df["District"] = df["District"].astype("Int64")
df["Ward"] = df["Ward"].astype("Int64")
df["Community Area"] = df["Community Area"].astype("Int64")

In [21]:
def recode_crime(x):
    if x in ["HOMICIDE", "ASSAULT", "BATTERY", "ROBBERY",
             "KIDNAPPING", "INTIMIDATION", "STALKING"]:
        return "VIOLENT CRIME"
    elif x in ["CRIM SEXUAL ASSAULT", "CRIMINAL SEXUAL ASSAULT",
               "SEX OFFENSE", "HUMAN TRAFFICKING"]:
        return "SEXUAL / TRAFFICKING"
    elif x in ["THEFT", "BURGLARY", "MOTOR VEHICLE THEFT",
               "CRIMINAL DAMAGE", "CRIMINAL TRESPASS",
               "DECEPTIVE PRACTICE", "ARSON"]:
        return "PROPERTY CRIME"
    elif x in ["NARCOTICS", "OTHER NARCOTIC VIOLATION"]:
        return "DRUG OFFENSE"
    elif x in ["PROSTITUTION", "GAMBLING", "LIQUOR LAW VIOLATION",
               "PUBLIC INDECENCY", "OBSCENITY", "RITUALISM",
               "PUBLIC PEACE VIOLATION"]:
        return "PUBLIC ORDER"
    elif x in ["WEAPONS VIOLATION", "CONCEALED CARRY LICENSE VIOLATION"]:
        return "WEAPONS OFFENSE"
    elif x in ["OFFENSE INVOLVING CHILDREN"]:
        return "CRIMES AGAINST CHILDREN"
    elif x in ["INTERFERENCE WITH PUBLIC OFFICER"]:
        return "LAW ENFORCEMENT RELATED"
    elif x in ["OTHER OFFENSE"]:
        return "MISCELLANEOUS CRIME"
    elif x in ["NON-CRIMINAL"]:
        return "NON-CRIMINAL"
    else:
        return "UNCLASSIFIED"

df["Crime Category"] = df["Primary Type"].apply(recode_crime)

Aggregate per hour

In [22]:
hourly_counts = df.groupby(["Crime Category", "incident_hour"]) \
                  .size() \
                  .unstack(fill_value=0)

print(hourly_counts)

incident_hour              0     1     2     3     4     5     6     7     8   \
Crime Category                                                                  
CRIMES AGAINST CHILDREN   510    28    20    15     8    11    23    41    72   
DRUG OFFENSE              133    65    30    22    13    19    30    56   135   
LAW ENFORCEMENT RELATED    18     7     8     7     2     4     1     2     1   
MISCELLANEOUS CRIME       996   314   254   165   142   153   197   368   587   
NON-CRIMINAL                0     0     0     0     0     0     0     0     0   
PROPERTY CRIME           8344  3021  2771  2433  2053  1873  2075  2587  3574   
PUBLIC ORDER               67    30    20    11     6     7    11    12    19   
SEXUAL / TRAFFICKING      483   108   117    93    66    57    60    63    78   
VIOLENT CRIME            3996  2912  2524  1904  1551  1245  1294  1677  2243   
WEAPONS OFFENSE           733   525   424   257   203   106    82    83    99   

incident_hour              

 Crime arrest rate

In [23]:
crime_rate = df.groupby("Crime Category")["Arrest"] \
               .mean() \
               .reset_index(name="arrest_rate")

print(crime_rate)

            Crime Category  arrest_rate
0  CRIMES AGAINST CHILDREN     0.100658
1             DRUG OFFENSE     0.988379
2  LAW ENFORCEMENT RELATED     0.843949
3      MISCELLANEOUS CRIME     0.106863
4             NON-CRIMINAL     0.250000
5           PROPERTY CRIME     0.042407
6             PUBLIC ORDER     0.601472
7     SEXUAL / TRAFFICKING     0.067517
8            VIOLENT CRIME     0.120557
9          WEAPONS OFFENSE     0.626435


Domestic crime rate

In [24]:
domestic_counts = df.groupby(["Crime Category", "Domestic"]) \
                    .size() \
                    .unstack(fill_value=0)

domestic_counts["domestic_rate"] = domestic_counts[True] / (domestic_counts[True] + domestic_counts[False])

print(domestic_counts)

Domestic                 False   True  domestic_rate
Crime Category                                      
CRIMES AGAINST CHILDREN    341   1636       0.827516
DRUG OFFENSE              5413      8       0.001476
LAW ENFORCEMENT RELATED    306      8       0.025478
MISCELLANEOUS CRIME       7867   6048       0.434639
NON-CRIMINAL                 3      1       0.250000
PROPERTY CRIME           94957   8918       0.085853
PUBLIC ORDER               898     53       0.055731
SEXUAL / TRAFFICKING      1984    682       0.255814
VIOLENT CRIME            39641  30384       0.433902
WEAPONS OFFENSE           9089     58       0.006341


In [25]:
end = time.time()
ram = df.memory_usage(deep=True).sum() / 1024**2

In [26]:
print("Execution time:", round(end - start, 2), "seconds")
print("Memory Usage:", round(ram, 2), "MB")

Execution time: 2.08 seconds
Memory Usage: 72.21 MB
